In [ ]:
!pip install -q langchain-openai langchain-core requests

In [ ]:
from langchain_huggingface import HuggingFaceHub
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b
print(multiply.invoke({'a':3, 'b':4}))

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
llm = HuggingFacePipeline.from_model_id(
    model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    task='text-generation',
    pipeline_kwargs=dict(
        temperature=0.5,
        max_new_tokens=100
    )
)
model = ChatHuggingFace(llm=llm)

## TOOL BINDING


In [ ]:
llm_with_tools = model.bind_tools([multiply])

from transformers import load_tool

controlnet_transformer = load_tool("diffusers/controlnet-canny-tool")
upscaler = load_tool("diffusers/latent-upscaler-tool")

In [ ]:
query = HumanMessage('can you multiply 3 with 1000')

In [ ]:
messages = [query]
messages

In [ ]:
result = llm_with_tools.invoke(messages)
result

In [24]:
messages.append(result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<|user|>\ncan you multiply 3 with 1000</s>\n<|assistant|>\nYes, you can multiply 3 with 1000 using the following formula:\n\n3 x 1,000 = 3,000.\n\nSo, 3 * 1,000 = 3,000.', additional_kwargs={}, response_metadata={}, id='run--2edd93d6-0a85-4bc6-8a5a-e72f112241dd-0')]

In [ ]:

tool_result = multiply.invoke(result.tool_calls)
# tool_result = result.tool_calls
tool_result

In [25]:
messages.append(tool_result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<|user|>\ncan you multiply 3 with 1000</s>\n<|assistant|>\nYes, you can multiply 3 with 1000 using the following formula:\n\n3 x 1,000 = 3,000.\n\nSo, 3 * 1,000 = 3,000.', additional_kwargs={}, response_metadata={}, id='run--2edd93d6-0a85-4bc6-8a5a-e72f112241dd-0'),
 []]

In [26]:
llm_with_tools.invoke(messages).content

NotImplementedError: Unsupported message type: <class 'list'>
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/MESSAGE_COERCION_FAILURE 

## TOOLS CREATE

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

res = get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

print(res)

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg] #llm do not feed this argument ,i will get it from the previous tool
            ) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1757289601, 'time_last_update_utc': 'Mon, 08 Sep 2025 00:00:01 +0000', 'time_next_update_unix': 1757376001, 'time_next_update_utc': 'Tue, 09 Sep 2025 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 88.2484}


In [30]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [31]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [32]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [33]:
ai_message = llm_with_tools.invoke(messages)

In [34]:
messages.append(ai_message)


In [35]:
ai_message.tool_calls

[]

In [36]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)


In [37]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='<|user|>\nWhat is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd</s>\n<|assistant|>\nIn India, the INR (Indian Rupee) is a currency unit used in the country. The INR is divided into 100 paise, and each paise is called a "pin." The conversion factor between INR and USD is 76.93, which means that one INR is equivalent to approximately 0.7693 USD.\n\nTo convert 10 inr to usd, you need to multiply ', additional_kwargs={}, response_metadata={}, id='run--7525d20d-2d7f-4e0f-9994-9f0fd24a0ffd-0')]